# LeetCode #1289: Minimum Falling Path Sum II

https://leetcode.com/problems/minimum-falling-path-sum-ii/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n^3)$ | $O(n^2)$ |
| **Optimal: Two-Min DP ★** | $O(n^2)$ | $O(1)$ |

---

## Understanding the Methods

### Brute Force
For each cell in each row, scan the entire previous row (excluding the same column) to find the minimum. This nested column scan leads to $O(n^3)$ time.

### Optimal: Two-Min DP ★
Track only the smallest and second-smallest values (and their indices) of the previous row. Each cell in the current row uses the global minimum from the previous row — unless that minimum shares the same column, in which case use the second minimum. Reduces per-row work from $O(n^2)$ to $O(n)$.

**Constraints:**
* `n == grid.length == grid[i].length`
* `1 <= n <= 200`
* `-99 <= grid[i][j] <= 99`

## Solutions

### C#

In [ ]:
public class Solution {
    public int MinFallingPathSum(int[][] grid) {
        int n = grid.Length;

        // Initialise prev-row extrema from the first row
        int min1 = int.MaxValue, min2 = int.MaxValue, minCol = -1;
        for (int c = 0; c < n; c++) {
            if (grid[0][c] < min1) { min2 = min1; min1 = grid[0][c]; minCol = c; }
            else if (grid[0][c] < min2) { min2 = grid[0][c]; }
        }

        for (int r = 1; r < n; r++) {
            int newMin1 = int.MaxValue, newMin2 = int.MaxValue, newCol = -1;
            for (int c = 0; c < n; c++) {
                // Can't use the direct-above column's minimum value
                int prev = (c == minCol) ? min2 : min1;
                int cur = grid[r][c] + prev;
                if (cur < newMin1) { newMin2 = newMin1; newMin1 = cur; newCol = c; }
                else if (cur < newMin2) { newMin2 = cur; }
            }
            min1 = newMin1; min2 = newMin2; minCol = newCol;
        }
        return min1;
    }
}

### Python

In [ ]:
class Solution:
    def minFallingPathSum(self, grid: list[list[int]]) -> int:
        n = len(grid)
        INF = float('inf')

        # Extract the two smallest values and the column of the smallest from row 0
        min1, min2, min_col = INF, INF, -1
        for c, v in enumerate(grid[0]):
            if v < min1: min2, min1, min_col = min1, v, c
            elif v < min2: min2 = v

        for r in range(1, n):
            new_min1, new_min2, new_col = INF, INF, -1
            for c in range(n):
                # Blocked from using the same column that produced min1
                prev = min2 if c == min_col else min1
                cur = grid[r][c] + prev
                if cur < new_min1: new_min2, new_min1, new_col = new_min1, cur, c
                elif cur < new_min2: new_min2 = cur
            min1, min2, min_col = new_min1, new_min2, new_col

        return min1

### Go

In [ ]:
func minFallingPathSum(grid [][]int) int {
    n := len(grid)
    const INF = 1<<30

    min1, min2, minCol := INF, INF, -1
    for c, v := range grid[0] {
        if v < min1 { min2, min1, minCol = min1, v, c } else if v < min2 { min2 = v }
    }

    for r := 1; r < n; r++ {
        newMin1, newMin2, newCol := INF, INF, -1
        for c := 0; c < n; c++ {
            // Forbidden to drop from the column holding the global previous minimum
            prev := min1
            if c == minCol { prev = min2 }
            cur := grid[r][c] + prev
            if cur < newMin1 { newMin2, newMin1, newCol = newMin1, cur, c } else if cur < newMin2 { newMin2 = cur }
        }
        min1, min2, minCol = newMin1, newMin2, newCol
    }
    return min1
}

### Rust

In [ ]:
impl Solution {
    pub fn min_falling_path_sum(grid: Vec<Vec<i32>>) -> i32 {
        let n = grid.len();
        let inf = i32::MAX / 2;

        let (mut min1, mut min2, mut min_col) = (inf, inf, usize::MAX);
        for (c, &v) in grid[0].iter().enumerate() {
            if v < min1 { min2 = min1; min1 = v; min_col = c; }
            else if v < min2 { min2 = v; }
        }

        for r in 1..n {
            let (mut nm1, mut nm2, mut nc) = (inf, inf, usize::MAX);
            for (c, &v) in grid[r].iter().enumerate() {
                // Use second minimum when current column matches the global minimum's column
                let prev = if c == min_col { min2 } else { min1 };
                let cur = v + prev;
                if cur < nm1 { nm2 = nm1; nm1 = cur; nc = c; }
                else if cur < nm2 { nm2 = cur; }
            }
            min1 = nm1; min2 = nm2; min_col = nc;
        }
        min1
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `grid = [[1,2,3],[4,5,6],[7,8,9]]`
Row 0 mins: min1=1(col0), min2=2. Row 1: col0 blocked → use min2=2: 4+2=6; col1: 5+1=6; col2: 6+1=7. New min1=6(col0). Row 2: col0 blocked → min2=6: 7+6=13; col1: 8+6=14; col2: 9+6=15. Answer: **13**.

### 2. Slightly Complex
**Input:** `grid = [[7]]`
Single cell, one row. Answer: **7**.

### 3. Edge Case: Time Factor
**Input:** $200 \times 200$ grid.
Two-min tracking reduces each row to a single $O(n)$ scan; total $O(n^2) = 40000$ operations.

### 4. Edge Case: Space Factor
**Input:** Any $200 \times 200$ grid.
Only six scalars (`min1, min2, minCol` plus `new*`) stored per row — $O(1)$ extra space; grid modified in-place if needed.

### 5. Almost-Impossible but Plausible
**Input:** All values $= -99$ in a $200 \times 200$ grid.
Every path has the same sum $200 \times (-99) = -19800$. The two-min tracker handles tie-breaking correctly; any column's second-minimum also equals $-99$.